# TUTORIAL: Next Sentence Prediction with BERT

- BOOK: [Dive Into DL](https://d2l.ai/chapter_natural-language-processing-pretraining/bert.html)
- NOTEBOOK: [Finetune BERT with IMDB](https://github.com/ufdatastudio/predictions/blob/development/notebook_experiments/tutorial-text_classification_hf-base.ipynb)
- NOTEBOOK: [Finetune BERT with TOLSA-M](https://github.com/ufdatastudio/predictions/blob/development/notebook_experiments/tutorial-text_classification_hf-tolsa_m.ipynb)

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

## Input Representation

BERT's text input type is both single and multiple

1. Single Text Input $ \rightarrow $ Tasks with Examples
2. Multiple Text Input $ \rightarrow $ Tasks with Examples
3. CODE: Get Tokens and Segments

### Single Text Input Tasks $ \rightarrow $ Tasks with Examples

| # | Task | Input Example | Output Example |
|---|------|---------------|----------------|
| 1 | Sentiment Analysis | "I absolutely loved this movie." | Positive |
| 2 | Topic Classification | "The Federal Reserve announced a change in interest rates." | Finance |
| 3 | Text Tagging (Named Entity Recognition) | "Barack Obama was born in Hawaii." | Barack Obama → PERSON, Hawaii → LOCATION |
| 4 | Language Identification | "Bonjour tout le monde." | French |
| 5 | Spam Detection | "Congratulations! You have won a free iPhone." | Spam |
| 6 | Emotion Classification | "I am so excited for graduation!" | Joy |
| 7 | Toxicity Detection | "You are terrible at everything." | Toxic |

---

Single: Concatenation of the special classification token `<cls>`, tokens of a text sequence, and the special separation token `<sep>`.

<div align="center"><b>Task: 1 (SA)</b></div>

BERT Input Sequence 1:
    ```
    [CLS] I absolutely loved this movie . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0
    ```

BERT Input Sequence 2:
    ```
    [CLS] The film was a complete waste of time . [SEP]
    ```
    
Segment IDs:
    ```
    0 0 0 0 0 0 0 0 0 0
    ```

### Multiple Text Input Tasks $ \rightarrow $ Tasks with Examples

| # | Task | Input Example 1 | Input Example 2 | Output Example |
|---|------|-----------------|-----------------|----------------|
| 1 | Natural Language Inference (NLI) | Premise: "A man is playing guitar." | Hypothesis: "A person is making music." | Entailment |
| 2 | Semantic Textual Similarity | Sentence 1: "The cat sat on the mat." | Sentence 2: "A cat is sitting on a rug." | Similarity = High |
| 3 | Duplicate Question Detection | Question 1: "How do I learn Python?" | Question 2: "What's the best way to study Python?" | Duplicate |
| 4 | Paraphrase Identification | Sentence 1: "The meeting starts at 9." | Sentence 2: "The meeting begins at 9." | Paraphrase |
| 5 | Information Retrieval / Document Ranking | Query: "Symptoms of diabetes" | Document: "Common symptoms include thirst and fatigue." | Relevant |
| 6 | Question Answering | Context: "Paris is the capital of France." | Question: "What is the capital of France?" | Paris |
| 7 | Dialogue Response Selection | User Message: "What's the weather today?" | Candidate Response: "It will be sunny and 75°F." | Appropriate Response |

Multiple: Concatenation of `<cls>`, tokens of the first text sequence, `<sep>`, tokens of the second text sequence, and `<sep>`.

<div align="center"><b>Task: 1 (NLI) — Pair 1</b></div>

BERT Input Sequence Pair 1:
    ```
    [CLS] A man is playing guitar . [SEP] A person is making music . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0 1 1 1 1 1 1 1
    ```

BERT Input Sequence Pair 2:
    ```
    [CLS] A woman is running outside . [SEP] A person is exercising . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0 1 1 1 1 1 1
    ```

### CODE: Get Tokens and Segments

In [2]:
# @save
def get_tokens_and_segments(tokens_a, tokens_b=None):
    """Get tokens of the BERT input sequence and their segment IDs."""
    tokens = ['<cls>'] + tokens_a + ['<sep>']
    # 0 and 1 are marking segment A and B, respectively
    segments = [0] * (len(tokens_a) + 2)
    if tokens_b is not None:
        tokens += tokens_b + ['<sep>']
        segments += [1] * (len(tokens_b) + 1)
    return tokens, segments

#### Image of BERT across input + token embeddings + segment embeddings + positional embeddings

![alt text](<Screenshot 2026-09-19 at 13.14.50.png>)

## CODE: Transformer Encoder + BERT Encoder

> Transformer Encoder Architecture: MultiheadAttention $ \rightarrow $ Feed Forward Network $ \rightarrow $ Layer Normalization (2x) $ \rightarrow $ Dropout

> BERT Encoder Architecture: Token Embeddings $ \rightarrow $ Segment Embeddings $ \rightarrow $ Transformer Encoder $ \rightarrow $ Positional Embeddings

> CODE: Transformer Block

- Becuase I'm getting `AttributeError: module 'd2l.torch' has no attribute 'TransformerEncoderBlock'`, so UF's claude-sonnect-4.6 generated the below.

In [3]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias=False):
        super().__init__()
        self.attention = nn.MultiheadAttention(num_hiddens, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(num_hiddens, ffn_num_hiddens),
            nn.ReLU(),
            nn.Linear(ffn_num_hiddens, num_hiddens)
        )
        self.norm1 = nn.LayerNorm(num_hiddens)
        self.norm2 = nn.LayerNorm(num_hiddens)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, valid_lens=None):
        attn_out, _ = self.attention(X, X, X)
        X = self.norm1(X + self.dropout(attn_out))
        ffn_out = self.ffn(X)
        return self.norm2(X + self.dropout(ffn_out))

# Monkey-patch it onto d2l
d2l.TransformerEncoderBlock = TransformerEncoderBlock

In [4]:
#@save
class BERTEncoder(nn.Module):
    """BERT encoder.
    
    Defines its architectual components in `__init__()`:
        embedding layers + stack of Tranform encoder blocks + learnable embedding while using instance attributes
    Defines the foward method (shows how data flows)

    """
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads,
                 num_blks, dropout, max_len=1000, **kwargs):
        super(BERTEncoder, self).__init__(**kwargs)
        
        """ Embedding layers """
        self.token_embedding = nn.Embedding(vocab_size, num_hiddens) # word level (token) embedding
        self.segment_embedding = nn.Embedding(2, num_hiddens) # distinguishing text pairs

        """Transformer encoder blocks """
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", d2l.TransformerEncoderBlock(
                num_hiddens, ffn_num_hiddens, num_heads, dropout, True))

        """ Learnable embedding layer """
        # In BERT, positional embeddings are learnable, thus we create a
        # parameter of positional embeddings that are long enough
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len,
                                                      num_hiddens))

    def forward(self, tokens, segments, valid_lens):
        # Shape of `X` remains unchanged in the following code snippet:
        # (batch size, max sequence length, `num_hiddens`)
        X = self.token_embedding(tokens) + self.segment_embedding(segments)
        X = X + self.pos_embedding[:, :X.shape[1], :]
        for blk in self.blks:
            X = blk(X, valid_lens)
        return X

> Create instance and initialize its parameters

In [5]:
vocab_size, num_hiddens, ffn_num_hiddens, num_heads = 10000, 768, 1024, 4
ffn_num_input, num_blks, dropout = 768, 2, 0.2
encoder = BERTEncoder(
    vocab_size, 
    num_hiddens, 
    ffn_num_hiddens, 
    num_heads,
    num_blks, dropout
    )

| Code | Meaning |
|------|---------|
| `torch.randint(0, vocab_size, (2, 8))` | Creates **2 sequences, each of length 8** — every value is a random integer index into the vocabulary (0 to 9999). This is where the "2 input sequences of length 8" comes from. |
| `(2, 8)` | The shape of `tokens`: **2** = number of BERT input sequences, **8** = number of tokens per sequence. |
| `[0, 0, 0, 0, 1, 1, 1, 1]` | Segment IDs for the **1st** BERT input sequence — first 4 tokens belong to segment A (`0`s), last 4 tokens belong to segment B (`1`s). This is a **multiple input** sequence (two sentences packed into one). |
| `[0, 0, 0, 1, 1, 1, 1, 1]` | Segment IDs for the **2nd** BERT input sequence — first 3 tokens belong to segment A (`0`s), last 5 tokens belong to segment B (`1`s). Also a **multiple input** sequence, just with a different split point. |
| `encoder(tokens, segments, None)` | Runs the forward pass — sums token, segment, and positional embeddings, then passes through the Transformer encoder blocks. `None` means no masking of valid lengths. |
| `encoded_X.shape` → `torch.Size([2, 8, 768])` | Output shape: **2** sequences × **8** tokens × **768** hidden units (`num_hiddens`). Every token in every sequence is now represented as a 768-dimensional vector [1]. |

--- 

> NOTE: `forward()` is never called directly. Instead, calling `encoder(tokens, segments, valid_lens)`
triggers `nn.Module.__call__()`, which internally invokes `forward()` — along with
any registered PyTorch hooks. Always call the model instance like a function, not `encoder.forward(...)`.

In [6]:
tokens = torch.randint(0, vocab_size, (2, 8))
segments = torch.tensor(
    [[0, 0, 0, 0, 1, 1, 1, 1],   # input sequence 1
    [0, 0, 0, 1, 1, 1, 1, 1]]    # input sequence 2
    )  
encoded_X = encoder(tokens, segments, None)
encoded_X.shape

torch.Size([2, 8, 768])

## CODE: MaskLM

> MaskLM $ \rightarrow $ MLP Architecture: Linear $ \rightarrow $ ReLU $ \rightarrow $ Layer Normalization $ \rightarrow $ Linear

- Why MLP?

In [7]:
#@save
class MaskLM(nn.Module):
    """The masked language model task of BERT."""
    def __init__(self, vocab_size, num_hiddens, **kwargs):
        super(MaskLM, self).__init__(**kwargs)
        self.mlp = nn.Sequential(nn.LazyLinear(num_hiddens),
                                 nn.ReLU(),
                                 nn.LayerNorm(num_hiddens),
                                 nn.LazyLinear(vocab_size))

    def forward(self, X, pred_positions):
        num_pred_positions = pred_positions.shape[1]
        pred_positions = pred_positions.reshape(-1)
        batch_size = X.shape[0]
        batch_idx = torch.arange(0, batch_size)
        # Suppose that `batch_size` = 2, `num_pred_positions` = 3, then
        # `batch_idx` is `torch.tensor([0, 0, 0, 1, 1, 1])`
        batch_idx = torch.repeat_interleave(batch_idx, num_pred_positions)
        masked_X = X[batch_idx, pred_positions]
        masked_X = masked_X.reshape((batch_size, num_pred_positions, -1))
        mlm_Y_hat = self.mlp(masked_X)
        return mlm_Y_hat

In [8]:
mlm = MaskLM(vocab_size, num_hiddens)
mlm_positions = torch.tensor([[1, 5, 2], [6, 1, 5]])
mlm_Y_hat = mlm(encoded_X, mlm_positions)
mlm_Y_hat.shape

torch.Size([2, 3, 10000])

In [9]:
mlm_Y = torch.tensor([[7, 8, 9], [10, 20, 30]])
loss = nn.CrossEntropyLoss(reduction='none')
mlm_l = loss(mlm_Y_hat.reshape((-1, vocab_size)), mlm_Y.reshape(-1))
mlm_l.shape

torch.Size([6])

## CODE: NextSentencePred

> Architecture: Linear (output)

In [10]:
#@save
class NextSentencePred(nn.Module):
    """The next sentence prediction task of BERT."""
    def __init__(self, **kwargs):
        super(NextSentencePred, self).__init__(**kwargs)
        self.output = nn.LazyLinear(2)

    def forward(self, X):
        # `X` shape: (batch size, `num_hiddens`)
        return self.output(X)

In [11]:
# PyTorch by default will not flatten the tensor as seen in mxnet where, if
# flatten=True, all but the first axis of input data are collapsed together
encoded_X = torch.flatten(encoded_X, start_dim=1)
# input_shape for NSP: (batch size, `num_hiddens`)
nsp = NextSentencePred()
nsp_Y_hat = nsp(encoded_X)
nsp_Y_hat.shape

torch.Size([2, 2])

In [12]:
nsp_y = torch.tensor([0, 1])
nsp_l = loss(nsp_Y_hat, nsp_y)
nsp_l.shape

torch.Size([2])

## CODE: BERT Model

> Architecture: BERTEncoder $ \rightarrow $ Hidden Layer $ \rightarrow $ MaskLM $ \rightarrow $ NextSentencePred

In [13]:
#@save
class BERTModel(nn.Module):
    """The BERT model."""
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens,
                 num_heads, num_blks, dropout, max_len=1000):
        super(BERTModel, self).__init__()
        self.encoder = BERTEncoder(vocab_size, num_hiddens, ffn_num_hiddens,
                                   num_heads, num_blks, dropout,
                                   max_len=max_len)
        self.hidden = nn.Sequential(nn.LazyLinear(num_hiddens),
                                    nn.Tanh())
        self.mlm = MaskLM(vocab_size, num_hiddens)
        self.nsp = NextSentencePred()

    def forward(self, tokens, segments, valid_lens=None, pred_positions=None):
        encoded_X = self.encoder(tokens, segments, valid_lens)
        if pred_positions is not None:
            mlm_Y_hat = self.mlm(encoded_X, pred_positions)
        else:
            mlm_Y_hat = None
        # The hidden layer of the MLP classifier for next sentence prediction.
        # 0 is the index of the '<cls>' token
        nsp_Y_hat = self.nsp(self.hidden(encoded_X[:, 0, :]))
        return encoded_X, mlm_Y_hat, nsp_Y_hat